In [1]:
import pandas as pd

df = pd.read_csv("data/unsplash_lite/photos.csv", sep="\t")

clean_df = df[[
    "photo_id",
    "photo_image_url",
    "photo_description",
    "photographer_username"
]].dropna(subset=["photo_image_url"])

clean_df = clean_df.rename(columns={
    "photo_id": "id",
    "photo_image_url": "url",
    "photo_description": "caption",
    "photographer_username": "author"
})

print(clean_df.head())

            id                                                url  \
0  oSf8ePoG9NU    https://images.unsplash.com/20/frozen-grass.JPG   
1  DlsOa5moK4w  https://images.unsplash.com/reserve/dRA4UuMBR2...   
2  XBGacbT3vXI  https://images.unsplash.com/photo-143633523196...   
3  FjikPptEbZg  https://images.unsplash.com/flagged/photo-1441...   
4  PXdBkNF8rlk  https://images.unsplash.com/photo-144596683821...   

                             caption        author  
0                                NaN     andrekoch  
1                  Fresh blueberries  majkmmiklavc  
2                                NaN    shooshanig  
3                                NaN      websanya  
4  Icebergs of Iceland’s Vatnajokull  andersjilden  


In [2]:
import requests

def check_url(url):
    try:
        r = requests.head(url, timeout=5)
        return r.status_code == 200
    except:
        return False

sample = clean_df.head(100)

valid = sample[sample["url"].apply(check_url)]

print("Valid:", len(valid), "/", len(sample))

Valid: 100 / 100


In [3]:
import torch
from transformers import CLIPProcessor, CLIPModel

MODEL_NAME = "openai/clip-vit-large-patch14"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

model.eval()

c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [4]:
import aiohttp
from PIL import Image
import io

async def load_image_from_url(session, url):
    try:
        async with session.get(url, timeout=10) as r:
            if r.status != 200:
                return None

            data = await r.read()
            return Image.open(io.BytesIO(data)).convert("RGB")

    except:
        return None

In [5]:
import torch
import numpy as np

def embed_images(model, processor, images):
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        features = model.get_image_features(**inputs)

    return features.cpu().numpy().astype("float32")

In [6]:
import aiohttp
import numpy as np
from tqdm import tqdm

async def build_embeddings(clean_df, limit=500):

    embeddings = []
    ids = []
    urls = []

    connector = aiohttp.TCPConnector(limit=5)

    async with aiohttp.ClientSession(connector=connector) as session:

        for _, row in tqdm(clean_df.head(limit).iterrows(), total=limit):

            img = await load_image_from_url(session, row["url"])
            if img is None:
                continue

            vec = embed_images(model, processor, [img])[0]

            embeddings.append(vec)
            ids.append(row["id"])
            urls.append(row["url"])

    return np.array(ids), np.array(embeddings, dtype="float32"), np.array(urls)

In [7]:
import faiss

def build_faiss(embeddings):
    faiss.normalize_L2(embeddings)  # ONLY HERE

    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)

    index.add(embeddings)

    return index

In [8]:
import pandas as pd
import numpy as np
import faiss

def save_system(index, ids, urls, embeddings):
    faiss.write_index(index, "index.faiss")

    pd.DataFrame({
        "id": ids,
        "url": urls
    }).to_csv("metadata.csv", index=False)

    np.save("embeddings.npy", embeddings)